In [1]:
%cd /content
!rm -rf HW-2_env
!git clone https://github.com/TebelevGt/HW-2_env.git
%cd HW-2_env
# Переключаемся на нужную ветку ПЕРЕД переходом в подпапку rl-shortest-path-agent
!git checkout HW_3_hybrid_rl
%cd rl-shortest-path-agent

/content
Cloning into 'HW-2_env'...
remote: Enumerating objects: 539, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 539 (delta 82), reused 165 (delta 67), pack-reused 354 (from 1)
Receiving objects: 100% (539/539), 11.77 MiB | 8.08 MiB/s, done.
Resolving deltas: 100% (248/248), done.
/content/HW-2_env
Branch 'HW_3_hybrid_rl' set up to track remote branch 'HW_3_hybrid_rl' from 'origin'.
Switched to a new branch 'HW_3_hybrid_rl'
/content/HW-2_env/rl-shortest-path-agent


In [15]:
from envs import ShortestPathDataset,get_shortest_path_dataset,PathVerifier
import networkx as nx

In [4]:
def generate_algorithmic_gold_trajectories(input_pkl: str, output_pkl: str):
    print(f"🚀 Генерация алгоритмических gold-траекторий для {input_pkl}...")
    dataset = ShortestPathDataset.load(input_pkl)

    for item in dataset.data:
        # Восстанавливаем граф
        G = nx.from_numpy_array(item.metadata["matrix"])
        start = item.metadata["start"]
        end = item.metadata["end"]

        # Находим оптимальный путь алгоритмически
        path = nx.shortest_path(G, source=start, target=end, weight="weight")

        # Генерируем идеальный текст <reasoning>
        reasoning_lines = [f"Starting at Node {start}."]

        for i in range(len(path) - 1):
            curr_node = path[i]
            next_node = path[i+1]
            weight = G[curr_node][next_node]["weight"]

            # Показываем, что модель "рассмотрела" соседей (берем реальных соседей из графа)
            neighbors = [f"Node {n} (weight {int(G[curr_node][n]['weight'])})" for n in G.neighbors(curr_node)]
            reasoning_lines.append(f"Neighbors of Node {curr_node} are: {', '.join(neighbors)}.")
            reasoning_lines.append(f"Moving to Node {next_node} with cost {int(weight)}.")

        reasoning_lines.append(f"Reached target Node {end}.")

        # Собираем финальный текст ответа
        reasoning_text = "\n".join(reasoning_lines)
        answer_text = ", ".join(map(str, path))

        full_gold_answer = f"<reasoning>\n{reasoning_text}\n</reasoning>\n<answer>{answer_text}</answer>"

        # Перезаписываем поле answer в объекте Data
        item.answer = full_gold_answer

    # Сохраняем обновленный датасет
    dataset.save(output_pkl)
    print(f"✅ Готово! Сохранено {len(dataset)} gold-задач в {output_pkl}")

# Использование для ваших файлов
generate_algorithmic_gold_trajectories("data/hard128k/hard_train_128.pkl", "data/hard128k/gold_train_128.pkl")

🚀 Генерация алгоритмических gold-траекторий для data/hard128k/hard_train_128.pkl...
Dataset saved: data/hard128k/gold_train_128.pkl (620 samples)
✅ Готово! Сохранено 620 gold-задач в data/hard128k/gold_train_128.pkl
